In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.ticker import MaxNLocator
from scipy.io import loadmat
from parametric_impact_pinn_tf1 import SequentialParametricImpactPINN
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'  # CPU:-1; GPU0: 0

In [ ]:
np.random.seed(1234)
tf.compat.v1.set_random_seed(1234)
print('TensorFlow version: {}'.format(tf.__version__))

## 1. Load reference data

In [ ]:
data = loadmat('all.mat')

# Cumulative impact event times from numerical simulation
impact_time_event = data['teout'][::2].ravel()

# Per-segment durations (relative)
impact_time_event_relative = np.zeros_like(impact_time_event)
impact_time_event_relative[0] = impact_time_event[0]
impact_time_event_relative[1:] = impact_time_event[1:] - impact_time_event[:-1]

print('Cumulative impact times (reference):', impact_time_event)
print('Per-segment durations  (reference):', impact_time_event_relative)

## 2. Define and run solver

In [ ]:
layers = [10, 64, 64, 64, 1]
n_impact = 6

solver = SequentialParametricImpactPINN(
    mx=1.0,
    my=0.3,
    k=1.0,
    c=0.0,
    D=1.0,
    r=1.0,
    layers=layers,
    hyp_ini_weight_loss=(1.0, 1.0, 10.0),
    optimizer_LB=True,
)

results = solver.run(
    x0=[[0.0]],
    xt0=[[0.0]],
    y0=[[0.0]],
    yt0=[[-1.0]],
    n_impact=n_impact,
    T_vector=[1, 5, 2, 5, 4, 10],
    nIter_vector=[1000, 1000, 1000, 1000, 1000, 1000],
    hyp_ini_para_vector=[0.5, 4, 1, 4, 3, 5],
    num_time_points=1000,
)

## 3. Extract results

In [ ]:
all_time = results['time']
all_x    = results['x']
all_xt   = results['xt']
all_xtt  = results['xtt']
all_y    = results['y']
all_yt   = results['yt']
all_ytt  = results['ytt']

impact_times       = solver.impact_times          # per-segment durations
impact_time_stamps = np.cumsum(impact_times)      # cumulative, for plot markers

print('Learned impact times (per segment):', np.round(impact_times, 4))
print('Cumulative (PINN):                 ', np.round(impact_time_stamps, 4))
print('Cumulative (reference):            ', np.round(impact_time_event, 4))

## 4. Plot — Deflection vs Time

In [ ]:
mpl.rcParams['font.family'] = 'Times New Roman'
mpl.rcParams['mathtext.fontset'] = 'custom'
mpl.rcParams['mathtext.rm'] = 'Times New Roman'
mpl.rcParams['mathtext.it'] = 'Times New Roman:italic'
mpl.rcParams['mathtext.bf'] = 'Times New Roman:bold'
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

fontsize_labels = 28
fontsize_ticks  = 28
fontsize_legend = 24
linewidth_plot  = 2
linewidth_vline = 1

plt.figure(figsize=(10, 4.5))
plt.plot(all_time, all_x, label=r'$x(t)$', linewidth=linewidth_plot)
plt.plot(all_time, all_y, label=r'$y(t)$', linestyle='--', linewidth=linewidth_plot)
for t_imp in impact_time_stamps:
    plt.axvline(x=t_imp, color='gray', linestyle='--', linewidth=linewidth_vline)
plt.xlabel('Time (t)', fontsize=fontsize_labels, labelpad=8)
plt.ylabel('Deflection (m)', fontsize=fontsize_labels, labelpad=12)
ax = plt.gca()
ax.xaxis.set_major_locator(MaxNLocator(nbins=10, prune='both'))
ax.yaxis.set_major_locator(MaxNLocator(nbins=5, prune='both'))
ax.tick_params(axis='both', labelsize=fontsize_ticks)
plt.legend(fontsize=fontsize_legend, loc='upper right')
plt.tight_layout(pad=1.5)
plt.xlim(0, impact_time_stamps[-1] * 1.05)
plt.ylim(-1.8, 1.8)
plt.savefig('deflection_vs_time.png', format='png', dpi=300)
plt.show()

## 5. Plot — Velocity vs Time

In [ ]:
plt.figure(figsize=(10, 4.5))
plt.plot(all_time, all_xt, label=r'$\dot{x}(t)$', linewidth=linewidth_plot)
plt.plot(all_time, all_yt, label=r'$\dot{y}(t)$', linestyle='--', linewidth=linewidth_plot)
for t_imp in impact_time_stamps:
    plt.axvline(x=t_imp, color='gray', linestyle='--', linewidth=linewidth_vline)
plt.xlabel('Time (t)', fontsize=fontsize_labels, labelpad=8)
plt.ylabel('Velocity (m/s)', fontsize=fontsize_labels, labelpad=12)
ax = plt.gca()
ax.xaxis.set_major_locator(MaxNLocator(nbins=10, prune='both'))
ax.yaxis.set_major_locator(MaxNLocator(nbins=5, prune='both'))
ax.tick_params(axis='both', labelsize=fontsize_ticks)
plt.legend(fontsize=fontsize_legend, loc='upper right')
plt.tight_layout(pad=1.5)
plt.xlim(0, impact_time_stamps[-1] * 1.05)
plt.ylim(-1.4, 1.4)
plt.savefig('velocity_vs_time.png', format='png', dpi=300)
plt.show()

## 6. Plot — Relative Deflection (x - y)

In [ ]:
plt.figure(figsize=(10, 4.5))
plt.plot(all_time, all_x - all_y, label='PINN', linestyle='--', linewidth=linewidth_plot)
plt.axhline(y= 1.0, color='black', linestyle='-.', linewidth=2)
plt.axhline(y=-1.0, color='black', linestyle='-.', linewidth=2)
for t_imp in impact_time_stamps:
    plt.axvline(x=t_imp, color='gray', linestyle='--', linewidth=linewidth_vline)
plt.xlabel('Time (t)', fontsize=fontsize_labels, labelpad=8)
plt.ylabel('Relative Deflection (m)', fontsize=fontsize_labels, labelpad=10)
ax = plt.gca()
ax.xaxis.set_major_locator(MaxNLocator(nbins=10, prune='both'))
ax.yaxis.set_major_locator(MaxNLocator(nbins=5, prune='both'))
ax.tick_params(axis='both', labelsize=fontsize_ticks)
plt.legend(fontsize=fontsize_legend, loc='lower right')
plt.tight_layout(pad=1.5)
plt.xlim(0, impact_time_stamps[-1] * 1.05)
plt.ylim(-1.4, 1.4)
plt.savefig('relative_deflection.png', format='png', dpi=300)
plt.show()

## 7. Plot — Impulse

In [ ]:
my = solver.my
delta_yt = np.diff(all_yt, axis=0)
time_for_delta_yt = all_time[:-1]
impulse = my * delta_yt

plt.figure(figsize=(10, 4.5))
plt.plot(time_for_delta_yt, impulse,
         label=r'$m(\dot{y}_{n+1} - \dot{y}_n)$', linewidth=linewidth_plot)
for t_imp in impact_time_stamps:
    plt.axvline(x=t_imp, color='gray', linestyle='--', linewidth=linewidth_vline)
plt.xlabel('Time (t)', fontsize=fontsize_labels, labelpad=8)
plt.ylabel('Impulse (Ns)', fontsize=fontsize_labels, labelpad=10)
ax = plt.gca()
ax.xaxis.set_major_locator(MaxNLocator(nbins=10, prune='both'))
ax.yaxis.set_major_locator(MaxNLocator(nbins=5, prune='both'))
ax.tick_params(axis='both', labelsize=fontsize_ticks)
plt.legend(fontsize=fontsize_legend)
plt.tight_layout(pad=1.5)
plt.xlim(0, impact_time_stamps[-1] * 1.05)
plt.ylim(-0.6, 0.6)
plt.savefig('impulse_plot.png', format='png', dpi=300)
plt.show()

## 8. Plot — Impact Time Convergence per Segment

In [ ]:
for i, model in enumerate(solver.models):
    # lambda_1_log[k] has shape (1,1); extract scalar per iteration
    lambda_log = [float(v[0, 0]) for v in model.lambda_1_log]

    plt.figure(figsize=(4.5, 4.5))
    plt.plot(lambda_log, linewidth=2, label='PINN')
    plt.axhline(y=impact_time_event_relative[i], color='black',
                linestyle='--', linewidth=2, label='Reference')
    plt.xlabel('Iteration', fontsize=24, labelpad=10)
    plt.ylabel('Impact time', fontsize=24, labelpad=10)
    ax = plt.gca()
    ax.xaxis.set_major_locator(MaxNLocator(nbins=4, prune='both'))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=4, prune='both'))
    ax.tick_params(axis='both', labelsize=24)
    legend_loc = 'upper right' if i in {3, 4, 5} else 'lower right'
    ax.legend(fontsize=20, loc=legend_loc)
    plt.tight_layout(pad=1.5)
    plt.savefig(f'impact_convergence_{i+1}.png', format='png', dpi=300)
    plt.show()

## 9. Plot — Training Loss (last segment)

In [ ]:
# To inspect a different segment, change the index: solver.models[i]
model = solver.models[-1]

fig, ax = plt.subplots(figsize=(5, 2.7))
ax.loglog(model.loss_log,     label='Total loss')
ax.loglog(model.loss_icx_log, label='Loss_icx')
ax.loglog(model.loss_fx_log,  label='Loss_fx')
ax.loglog(model.loss_f_log,   label='Loss_f')
ax.set_xlabel('Iteration')
ax.set_ylabel('Loss')
ax.legend()
plt.tight_layout()
plt.show()